# Speechocean762 탐색적 데이터 분석 (EDA)

## 목적
- 데이터 구조 파악
- 점수 분포 시각화 (accuracy, fluency, prosodic, total)
- 피처 간 상관관계 분석
- 논문 아이디어 검증: 각 피처가 총점에 얼마나 기여하는지 확인

## 데이터셋 소개
- 5,000개 영어 발화, 250명 비원어민 화자 (모국어: 중국어)
- 5명의 전문가가 독립적으로 채점
- 점수 레벨: 문장(sentence) / 단어(word) / 음소(phoneme)
- CC BY 4.0 라이센스 (상업적 사용 포함 무료)

## 0. 필요 라이브러리 설치
처음 실행할 때만 아래 셀을 실행하세요.

In [ ]:
# 필요한 라이브러리 설치
# 터미널에서 실행하거나 주석 해제 후 실행
# !pip install datasets pandas numpy matplotlib seaborn scikit-learn

## 1. 데이터 로드
HuggingFace에서 자동으로 다운로드됩니다. (첫 실행 시 약 1~2분 소요)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (선택사항 - 없어도 됨)
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")
print("데이터셋 다운로드 중... (첫 실행 시 1~2분 소요)")

# HuggingFace에서 speechocean762 로드
# train: 2500개, test: 2500개
dataset = load_dataset("mispeech/speechocean762")

print(f"\n✅ 데이터 로드 완료!")
print(f"  - Train 샘플 수: {len(dataset['train'])}")
print(f"  - Test  샘플 수: {len(dataset['test'])}")

## 2. 데이터 구조 파악
데이터가 어떻게 생겼는지 확인합니다.

In [ ]:
# 첫 번째 샘플 출력
sample = dataset['train'][0]

print("=" * 60)
print("[샘플 1개 구조]")
print("=" * 60)
print(f"발화 텍스트  : {sample['text']}")
print()
print("[문장 레벨 점수 (0~10)]")
print(f"  accuracy   (발음 정확도) : {sample['accuracy']}")
print(f"  completeness (완성도)   : {sample['completeness']}")
print(f"  fluency    (유창성)     : {sample['fluency']}")
print(f"  prosodic   (운율)       : {sample['prosodic']}")
print(f"  total      (총점)       : {sample['total']}")
print()
print("[단어 레벨 점수 (첫 번째 단어)]")
word = sample['words'][0]
print(f"  단어: {word['text']}")
print(f"  음소: {word['phones']}")
print(f"  음소별 정확도: {word['phones-accuracy']}")
print(f"  단어 accuracy: {word['accuracy']}")
print(f"  단어 stress  : {word['stress']}")
print(f"  단어 total   : {word['total']}")

In [ ]:
# pandas DataFrame으로 변환 (분석하기 쉽게)
def dataset_to_df(split):
    """
    HuggingFace 데이터셋을 pandas DataFrame으로 변환
    문장 레벨 점수만 추출 (단어/음소 레벨은 별도 분석)
    """
    records = []
    for item in split:
        records.append({
            'text'        : item['text'],
            'accuracy'    : item['accuracy'],      # 발음 정확도
            'completeness': item['completeness'],  # 완성도 (빠진 단어 없는지)
            'fluency'     : item['fluency'],       # 유창성 (멈춤, 속도)
            'prosodic'    : item['prosodic'],       # 운율 (강세, 억양)
            'total'       : item['total'],          # 전문가 총점
            'word_count'  : len(item['words']),     # 단어 수
        })
    return pd.DataFrame(records)

train_df = dataset_to_df(dataset['train'])
test_df  = dataset_to_df(dataset['test'])
all_df   = pd.concat([train_df, test_df], ignore_index=True)

print("DataFrame 변환 완료")
print(f"전체 shape: {all_df.shape}")
print()
all_df.head()

## 3. 기본 통계량
각 점수의 분포를 수치로 확인합니다.

In [ ]:
# 기본 통계량
score_cols = ['accuracy', 'completeness', 'fluency', 'prosodic', 'total']

print("=" * 60)
print("[점수 기본 통계량]")
print("=" * 60)
stats = all_df[score_cols].describe().round(2)
print(stats)
print()

# 점수 범위 확인
print("[점수 범위]")
for col in score_cols:
    print(f"  {col:15s}: min={all_df[col].min():.1f}, max={all_df[col].max():.1f}, "
          f"mean={all_df[col].mean():.2f}, std={all_df[col].std():.2f}")

## 4. 점수 분포 시각화
각 평가 요소의 분포가 어떻게 생겼는지 확인합니다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Speechocean762: Score Distributions', fontsize=16, fontweight='bold')

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']
labels = {
    'accuracy'    : 'Accuracy (발음 정확도)',
    'completeness': 'Completeness (완성도)',
    'fluency'     : 'Fluency (유창성)',
    'prosodic'    : 'Prosodic (운율)',
    'total'       : 'Total Score (총점)'
}

for idx, (col, color) in enumerate(zip(score_cols, colors)):
    ax = axes[idx // 3][idx % 3]
    
    # 히스토그램
    ax.hist(all_df[col], bins=20, color=color, alpha=0.7, edgecolor='white')
    
    # 평균선
    mean_val = all_df[col].mean()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2,
               label=f'Mean: {mean_val:.2f}')
    
    ax.set_title(labels[col], fontsize=11)
    ax.set_xlabel('Score (0-10)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

# 마지막 빈 칸 숨기기
axes[1][2].set_visible(False)

plt.tight_layout()
plt.savefig('score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: score_distributions.png")

## 5. 🔑 핵심 분석: 각 피처와 총점의 상관관계

이 분석이 여러분 논문 아이디어의 핵심 근거가 됩니다.
- **상관계수가 높을수록** → 그 피처가 총점에 많이 기여한다는 뜻
- 목적별로 가중치를 다르게 주는 아이디어의 기초 수치

In [ ]:
from scipy import stats as scipy_stats

# 각 피처와 total 점수의 피어슨 상관계수
feature_cols = ['accuracy', 'completeness', 'fluency', 'prosodic']

print("=" * 60)
print("[각 피처 vs 총점 상관관계 (Pearson r)]")
print("=" * 60)
print(f"{'피처':<20} {'r값':>8} {'p-value':>12} {'의미'}")
print("-" * 60)

correlations = {}
for col in feature_cols:
    r, p = scipy_stats.pearsonr(all_df[col], all_df['total'])
    correlations[col] = r
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"  {col:<18} {r:>8.4f} {p:>12.2e} {sig}")

print()
print("※ *** p<0.001 (매우 유의미한 상관관계)")
print()
print("[해석]")
best = max(correlations, key=correlations.get)
worst = min(correlations, key=correlations.get)
print(f"  총점과 가장 상관 높은 피처: {best} (r={correlations[best]:.4f})")
print(f"  총점과 가장 상관 낮은 피처: {worst} (r={correlations[worst]:.4f})")

In [ ]:
# 상관관계 히트맵
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 상관계수 히트맵
corr_matrix = all_df[score_cols].corr()
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True  # 상삼각 마스킹

sns.heatmap(corr_matrix,
            annot=True, fmt='.3f',
            cmap='RdYlGn', vmin=0, vmax=1,
            ax=axes[0], linewidths=0.5,
            annot_kws={'size': 11})
axes[0].set_title('피처 간 상관관계 (Pearson r)', fontsize=12, fontweight='bold')

# 오른쪽: 각 피처 → total 상관계수 막대그래프
feat_labels = {
    'accuracy'    : 'Accuracy\n(발음정확도)',
    'completeness': 'Completeness\n(완성도)',
    'fluency'     : 'Fluency\n(유창성)',
    'prosodic'    : 'Prosodic\n(운율)'
}
r_vals = [correlations[c] for c in feature_cols]
bar_colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

bars = axes[1].bar(
    [feat_labels[c] for c in feature_cols],
    r_vals,
    color=bar_colors, alpha=0.8, edgecolor='white', linewidth=1.5
)

# 값 표시
for bar, val in zip(bars, r_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Pearson r (with Total Score)')
axes[1].set_title('각 피처 → 총점 기여도', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
axes[1].axhline(y=0.9, color='red', linestyle='--', alpha=0.5, label='r=0.9 기준선')
axes[1].legend()

plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: feature_correlation.png")

## 6. 점수 구간별 분석
학습자를 CEFR처럼 구간으로 나눠서 분포를 봅니다.

In [ ]:
# 총점을 3개 구간으로 나누기
# 0~4: 낮음 (Low), 5~7: 중간 (Mid), 8~10: 높음 (High)
def categorize_score(score):
    if score <= 4:
        return 'Low (0-4)'
    elif score <= 7:
        return 'Mid (5-7)'
    else:
        return 'High (8-10)'

all_df['level'] = all_df['total'].apply(categorize_score)

# 구간별 각 피처 평균
level_means = all_df.groupby('level')[feature_cols].mean().round(3)

print("=" * 60)
print("[레벨 구간별 피처 평균값]")
print("=" * 60)
print(level_means)
print()

# 구간별 샘플 수
print("[구간별 샘플 수]")
print(all_df['level'].value_counts().sort_index())

In [ ]:
# 레벨별 피처 프로파일 레이더 차트
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(feature_cols))
width = 0.25
level_order = ['Low (0-4)', 'Mid (5-7)', 'High (8-10)']
level_colors = ['#d62728', '#ff7f0e', '#2ca02c']

for i, (level, color) in enumerate(zip(level_order, level_colors)):
    if level in level_means.index:
        vals = level_means.loc[level, feature_cols].values
        bars = ax.bar(x + i * width, vals, width,
                      label=level, color=color, alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels(['Accuracy\n(발음정확도)', 'Completeness\n(완성도)',
                    'Fluency\n(유창성)', 'Prosodic\n(운율)'],
                   fontsize=10)
ax.set_ylabel('Average Score (0-10)')
ax.set_title('레벨 구간별 각 피처 평균 점수', fontsize=13, fontweight='bold')
ax.legend(title='총점 구간', fontsize=10)
ax.set_ylim(0, 12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('level_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: level_profile.png")

## 7. 🎯 논문 아이디어 연결: 목적별 가중치 시뮬레이션

핵심 아이디어: 같은 학습자라도 **목적에 따라 가중치를 다르게** 주면 점수가 달라진다.

예시:
- **여행 목적**: 발음(0.4) + 유창성(0.3) + 운율(0.2) + 완성도(0.1)
- **취업 목적**: 어휘/완성도(0.4) + 정확도(0.3) + 유창성(0.2) + 운율(0.1)
- **균등 (CEFR 기준)**: 모두 동일(0.25)

In [ ]:
# 목적별 가중치 정의 (논문에서 이 가중치 설정 근거가 핵심 contribution)
# 현재는 가설 기반 → 나중에 설문/전문가 인터뷰로 검증 필요

weights = {
    'equal': {
        # 균등 가중치 = 기존 CEFR 방식 (baseline)
        'accuracy': 0.25, 'completeness': 0.25,
        'fluency': 0.25, 'prosodic': 0.25
    },
    'travel': {
        # 여행 목적: 발음과 유창성 중시
        # 근거: 여행 시 의사소통 명확성이 핵심
        'accuracy': 0.40, 'completeness': 0.10,
        'fluency': 0.30, 'prosodic': 0.20
    },
    'business': {
        # 취업/비즈니스 목적: 정확도와 완성도 중시
        # 근거: 직장 내 의사소통은 정확한 표현이 중요
        'accuracy': 0.30, 'completeness': 0.40,
        'fluency': 0.20, 'prosodic': 0.10
    },
    'academic': {
        # 학업 목적: 운율과 완성도 중시
        # 근거: 발표/토론에서 전달력이 중요
        'accuracy': 0.20, 'completeness': 0.30,
        'fluency': 0.20, 'prosodic': 0.30
    }
}

# 가중 점수 계산
for purpose, w in weights.items():
    all_df[f'score_{purpose}'] = (
        all_df['accuracy']     * w['accuracy'] +
        all_df['completeness'] * w['completeness'] +
        all_df['fluency']      * w['fluency'] +
        all_df['prosodic']     * w['prosodic']
    )

score_compare_cols = ['total', 'score_equal', 'score_travel',
                      'score_business', 'score_academic']
print("[목적별 가중 점수 통계]")
print(all_df[score_compare_cols].describe().round(3))

In [ ]:
# 목적별 점수 분포 비교
fig, ax = plt.subplots(figsize=(12, 5))

purpose_labels = {
    'total'           : 'Expert Total\n(전문가 총점)',
    'score_equal'     : 'Equal Weight\n(균등 baseline)',
    'score_travel'    : 'Travel Purpose\n(여행 목적)',
    'score_business'  : 'Business Purpose\n(취업 목적)',
    'score_academic'  : 'Academic Purpose\n(학업 목적)'
}
colors_box = ['#2c7bb6', '#74c476', '#fd8d3c', '#e6550d', '#756bb1']

data_to_plot = [all_df[col].values for col in score_compare_cols]
bp = ax.boxplot(data_to_plot, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))

for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels([purpose_labels[c] for c in score_compare_cols], fontsize=9)
ax.set_ylabel('Score (0-10)')
ax.set_title('목적별 가중 점수 분포 비교\n(같은 학습자도 목적에 따라 점수가 달라짐)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('purpose_score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: purpose_score_comparison.png")

In [ ]:
# 목적에 따라 순위가 바뀌는 학습자 찾기
# → 이게 논문의 핵심 예시가 됩니다

# 전체 점수 기준 중간 구간 학습자 30명 샘플링
mid_learners = all_df[
    (all_df['total'] >= 5) & (all_df['total'] <= 7)
].sample(30, random_state=42)[['text', 'accuracy', 'fluency',
                                'score_travel', 'score_business', 'total']]

# 여행 vs 취업 점수 차이가 큰 학습자
mid_learners['gap_travel_vs_business'] = (
    mid_learners['score_travel'] - mid_learners['score_business']
).round(2)

# 차이가 큰 순으로 정렬
interesting = mid_learners.nlargest(5, 'gap_travel_vs_business')

print("=" * 70)
print("[흥미로운 케이스: 여행 점수 > 취업 점수 TOP 5]")
print("이런 학습자는 '여행용'으로는 충분하지만 '취업용'으로는 더 공부 필요")
print("=" * 70)
print(interesting.round(2).to_string(index=False))
print()

# 반대 케이스
interesting2 = mid_learners.nsmallest(5, 'gap_travel_vs_business')
print("[반대 케이스: 취업 점수 > 여행 점수 TOP 5]")
print("이런 학습자는 '취업용'으로는 OK지만 발음/유창성 개선 필요")
print(interesting2.round(2).to_string(index=False))

## 8. 요약 및 다음 단계

### EDA 결과 요약

In [ ]:
print("=" * 65)
print("EDA 결과 요약")
print("=" * 65)

print(f"\n1. 데이터 규모")
print(f"   총 {len(all_df)}개 발화, 문장/단어/음소 3단계 레이블")

print(f"\n2. 점수 분포")
for col in score_cols:
    print(f"   {col:<15}: 평균 {all_df[col].mean():.2f}, "
          f"중앙값 {all_df[col].median():.1f}")

print(f"\n3. 피처-총점 상관관계")
for col in feature_cols:
    r, _ = scipy_stats.pearsonr(all_df[col], all_df['total'])
    bar = '█' * int(r * 20)
    print(f"   {col:<15}: r={r:.4f} {bar}")

print(f"\n4. 목적별 가중치 시뮬레이션")
for purpose in ['equal', 'travel', 'business', 'academic']:
    col = f'score_{purpose}'
    print(f"   {purpose:<12}: 평균 {all_df[col].mean():.3f}, "
          f"std {all_df[col].std():.3f}")

print(f"\n5. 논문 아이디어 검증")
print("   ✅ 목적에 따라 같은 학습자의 점수가 달라짐")
print("   ✅ 여행/취업 목적 간 점수 순위 역전 케이스 존재")
print("   ✅ 이 데이터셋으로 baseline 실험 가능")

print(f"\n다음 단계")
print("   1. Bamdev(2023) 방식으로 ML baseline 모델 구현")
print("   2. 목적별 가중치 근거 수립 (설문 or 기존 루브릭 분석)")
print("   3. 목적 조건부 평가 모델 설계")
print("   4. 지도교수님께 중간 결과 공유")